# CyteOnto Call

Below are cells to run a CyteOnto call using the CyteOnto package

In [ ]:
from pathlib import Path

from cyteonto import CyteOntoConfig, run_cyteonto
from shared.repo import REPO_ROOT

cfg = CyteOntoConfig(h5adPath=REPO_ROOT / "output" / "cytetype" / "data" / "SRX17412841_cytetype_annotated.h5ad")
similarities = run_cyteonto(cfg)

In [ ]:
from cyteonto import check_pending_runs

results = check_pending_runs()
# results is a dict of {run_id: DataFrame} for every run that completed since last session
# to use one as the working DataFrame:
# similarities = results["run-<uuid>"]

# CyteOnto Analysis

Below, an analysis of the results returned by `run_cyteonto` (scripts/cyteonto) quantifies the agreement between STATE and CyteType labels. Make sure some CyteOnto runs have been completed above or check the `output/` directory, then set `run_id` below to the returned run ID.

In [ ]:
import math
from pathlib import Path

import matplotlib.pyplot as plt
import pandas as pd
import seaborn as sns

## Mean similarity by STATE annotation

In [ ]:
import scanpy as sc
from dotenv import load_dotenv
from storage import download_from_r2

from cyteonto import attach_cytescores_to_obs
from shared.repo import REPO_ROOT

load_dotenv()

SRX = "SRX17412841"
R2_PREFIX = "cytetype_pipeline_20260522_175813"
AUTHOR_COL = "cell_type"
ALGO_COL = "cytetype_annotation_leiden_merged"
ALGORITHM = "algo1"

run_id = "run-84e4f282-5897-4b18-af75-cda75134e61f"
similarities = pd.read_csv(REPO_ROOT / "output" / "cyteonto" / "runs" / f"{run_id}.csv")

local_h5ad = REPO_ROOT / "data" / "cyteonto_analysis" / f"{SRX}_annotated.h5ad"
download_from_r2(f"{R2_PREFIX}/{SRX}_annotated.h5ad", local_h5ad)
adata = sc.read_h5ad(local_h5ad)
obs = attach_cytescores_to_obs(
    adata.obs,
    similarities,
    author_col=AUTHOR_COL,
    algorithm_col=ALGO_COL,
    algorithm=ALGORITHM,
)
obs = obs.dropna(subset=["cytescore_similarity"])
similarities.head()

In [ ]:
summaries = obs.groupby(AUTHOR_COL)["cytescore_similarity"].describe()
mean_similarities = summaries["mean"].sort_values(ascending=False)

sns.boxplot(
    data=obs,
    x="cytescore_similarity",
    y=AUTHOR_COL,
    order=mean_similarities.index,
    fill=False,
    fliersize=0,
)

sns.pointplot(
    data=obs,
    x="cytescore_similarity",
    y=AUTHOR_COL,
    order=mean_similarities.index,
    linestyle="none",
    errorbar=None,
    marker="x",
)

plt.ylabel("STATE Label")
plt.xlabel("Mean Similarity to CyteType Label")
plt.xlim([0, 1.025])
plt.title(f"Mean Similarity of CyteType Labels\nby STATE Cluster ({SRX})")

summaries.sort_values(by="mean", ascending=False)

## Distributions of similarity scores by cluster and algorithm

In [ ]:
cols = 4

fig_state, axs_state = plt.subplots(ncols=cols, nrows=math.ceil(obs[AUTHOR_COL].nunique() / cols))
fig_cyte, axs_cyte = plt.subplots(ncols=cols, nrows=math.ceil(obs[ALGO_COL].nunique() / cols))

for i, label in enumerate(obs[AUTHOR_COL].unique()):
    axs_state[i // cols, i % cols].hist(obs.loc[obs[AUTHOR_COL] == label, "cytescore_similarity"])

for i, label in enumerate(obs[ALGO_COL].unique()):
    axs_cyte[i // cols, i % cols].hist(obs.loc[obs[ALGO_COL] == label, "cytescore_similarity"])

fig_state.suptitle("Distribution of Similarity Scores by STATE Cluster")
fig_cyte.suptitle("Distribution of Similarity Scores by CyteType Cluster")

fig_state.tight_layout()
fig_cyte.tight_layout()

## STATE label that best matches each CyteType cluster

In [ ]:
# Cluster approach: one CyteType label per cluster and one STATE label per cluster
# = the author label that is most abundant in cells assigned to that cluster.

_key = [ALGO_COL]


def _dominant_state_and_stats(g: pd.DataFrame) -> pd.Series:
    vc = g[AUTHOR_COL].value_counts()
    top = vc.index[0]
    n_top = int(vc.iloc[0])
    n = len(g)
    return pd.Series(
        {
            "stateLabel": top,
            "nCellsInCluster": n,
            "nDominantState": n_top,
            "fracDominantState": n_top / n if n else float("nan"),
        }
    )


_by_cluster = obs.groupby(_key, group_keys=True).apply(_dominant_state_and_stats, include_groups=False)
cluster_state_cytetype = _by_cluster.reset_index()
cluster_state_cytetype = cluster_state_cytetype.rename(columns={ALGO_COL: "cyteTypeLabel"})
cluster_state_cytetype = cluster_state_cytetype.sort_values("cyteTypeLabel", kind="stable")

print("CyteType cluster -> STATE label (most abundant author label in cluster):\n")
print(f"Number of CyteType clusters: {len(cluster_state_cytetype['cyteTypeLabel'].unique())}")
print(f"Number of STATE labels: {len(cluster_state_cytetype['stateLabel'].unique())}")
display(cluster_state_cytetype.sort_values(by="fracDominantState"))